In [41]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

print("Environment stable.")

Environment stable.


## **1. Load the JCI ESS document**

In [42]:
JCI_PDF = "/Users/siri/Desktop/sghpolicyreview/data/ESS Standards_7th Ed .pdf"

from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(JCI_PDF)
pages = loader.load()

print(f"Total pages loaded = {len(pages)}")


Total pages loaded = 406


## **2. Grouping pages by chapter**

In [43]:
import re
from collections import defaultdict

VALID_CODES = {
    "APR","IPSG","ACC","PCC","AOP","COP","ASC","MMU",
    "QPS","PCI","GLD","FMS","SQE","MOI",
    "MPE","HRP"
}

def detect_chapter_from_text(text: str):
    lines = text.split("\n")
    for line in lines:
        line = line.strip()
        match = re.match(r'^.+\(([A-Z]{3,4})\)\s*$', line)
        if match:
            code = match.group(1)               
            if code in VALID_CODES:
                return code
    return None

# ── STEP 1: Build chapter_pages first ────────────────────────────────
chapter_pages = defaultdict(list)
current_chapter = "UNKNOWN"

for page in pages:
    detected = detect_chapter_from_text(page.page_content)
    if detected:
        current_chapter = detected
    chapter_pages[current_chapter].append(page)

# Print result
# total = 0
# for chapter, pg in chapter_pages.items():
#     print(f"{chapter}: {len(pg)} pages")
#     total += len(pg)
# print("Total grouped pages:", total)

# ── STEP 2: NOW clean end-of-book pages from HRP ─────────────────────
# The glossary/appendix pages at the end of the PDF get incorrectly 
# assigned to HRP because there's no new chapter title after it

END_OF_BOOK_MARKERS = ["glossary", "index", "appendix", "references"]

def is_end_of_book_page(page_text):
    first_200 = page_text.strip()[:200].lower()
    return any(marker in first_200 for marker in END_OF_BOOK_MARKERS)

if "HRP" in chapter_pages:
    cleaned_hrp = []
    removed = 0
    for page in chapter_pages["HRP"]:
        if is_end_of_book_page(page.page_content):
            removed += 1
        else:
            cleaned_hrp.append(page)
    chapter_pages["HRP"] = cleaned_hrp
    # print(f"\nRemoved {removed} end-of-book pages from HRP")
    # print(f"HRP now has {len(chapter_pages['HRP'])} pages")

    print("=== CHAPTER PAGE COUNTS ===")
for chapter, pg in chapter_pages.items():
    print(f"  {chapter}: {len(pg)} pages")
print(f"\nTotal pages: {total}")
# print(f"HRP after cleanup: {len(chapter_pages['HRP'])} pages (glossary removed)")

=== CHAPTER PAGE COUNTS ===
  UNKNOWN: 2 pages
  FMS: 42 pages
  APR: 18 pages
  ACC: 22 pages
  PCC: 21 pages
  AOP: 30 pages
  COP: 32 pages
  ASC: 16 pages
  QPS: 15 pages
  PCI: 27 pages
  GLD: 32 pages
  SQE: 28 pages
  MOI: 25 pages
  MPE: 13 pages
  HRP: 32 pages
  IPSG: 16 pages
  MMU: 22 pages

Total pages: 15


In [44]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

def build_structured_chapter_chunks(chapter_pages, chunk_size=800, chunk_overlap=150):
    
    inner_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", " "]
    )

    chapter_chunks = {}

    for chapter, pg_list in chapter_pages.items():
        
        # 1. Merge all pages into one string
        full_text = "\n".join([p.page_content for p in pg_list])

        # 2. Clean page artifacts
        full_text = re.sub(r'Page\s+\d+\s+of\s+\d+', '', full_text) # clears the page no.s
        full_text = re.sub(r'\n{3,}', '\n\n', full_text) # clears the unecesssary white spaces

        # 3. Skip the TOC/summary section at the top
        # The real content starts at "Standards, Intents, and Measurable Elements"
        # which appears as a heading before the first real standard
        toc_marker = re.search(
            r'Standards,\s+Intents,\s+and\s+Measurable\s+Elements', # finds "Standards", then one or more whitespaces. then finds "Intents", then one or more whitespaces and so on.
            full_text,
            re.IGNORECASE # ignores upper and lower cases
        )
        if toc_marker:
            full_text = full_text[toc_marker.start():] # gives the character from the text where the match begins. deletes everyhting before this character and saves the content from that point onwards.
        
        # 4. Split by standard codes that BELONG to this chapter only
        # pattern = rf'(?:\n|^)Standard\s+({chapter}\.\d+(?:\.\d+)?)\b'  

        # start of the line/text, "Standard" followed by one/more whitespace, chapter code, one or more digits, word boundary-stops matching mid word.
        # \s* means zero or more spaces between chapter code and the dot
        # this handles both "ACC.1" and "AOP .1"
        # pattern = rf'(?:\n|^)Standard\s+({chapter}\s*\.\s*\d+(?:\s*\.\s*\d+)?)\b'

        # NEW — matches "Standard ACC.1", "Standard AOP .1", and "Requirement APR.1"
        pattern = rf'(?:\n|^)(?:Standard|Requirement)\s+({chapter}\s*\.\s*\d+(?:\s*\.\s*\d+)?)\b'
        splits = re.split(pattern, full_text) # this gives alternate code and content splits, eg: splits[0]=intro content before standards, splits[1]=code, splits[2]=content

        docs = []

        # splits[0] is intro text before any standard code — keep it
        if splits[0].strip():
            docs.append(Document(
                page_content=splits[0].strip(),
                metadata={"chapter": chapter, "standard_code": "INTRO"}
            ))

        for i in range(1, len(splits), 2):  #stepping through by 2 as the code and content alternates. 
            if i + 1 >= len(splits): #This guard prevents crashing if the list ends on a code with no content after it.
                break

            # standard_code = splits[i].strip()
            standard_code = splits[i].strip().replace(" ", "")
            content = f"{standard_code}\n{splits[i+1]}"

            if len(content.strip()) < 100: # if character size is. < 100 then skip as it cannot be a whole standard can be for eg: "See ACC.5.1 also."
                continue

            if len(content) <= chunk_size: # If the standard's content fits within chunk_size characters then it is kept as one single chunk. Short standards like ACC.4 stay intact.
                docs.append(Document(
                    page_content=content.strip(),
                    metadata={
                        "chapter": chapter,
                        "standard_code": standard_code
                    }
                ))
            else:
                sub_chunks = inner_splitter.split_text(content)
                for j, chunk_text in enumerate(sub_chunks):
                    if len(chunk_text.strip()) < 100:
                        continue
                    docs.append(Document(
                        page_content=chunk_text.strip(),
                        metadata={
                            "chapter": chapter,
                            "standard_code": standard_code,
                            "chunk_index": j # sub chunks given an index
                        }
                    ))

        chapter_chunks[chapter] = docs
        print(f"{chapter}: {len(docs)} chunks across "
              f"{len(set(d.metadata['standard_code'] for d in docs))} standards")

    return chapter_chunks


chapter_chunks = build_structured_chapter_chunks(
    chapter_pages,
    chunk_size=1000,
    chunk_overlap=250
)

UNKNOWN: 1 chunks across 1 standards
FMS: 131 chunks across 26 standards
APR: 32 chunks across 12 standards
ACC: 80 chunks across 17 standards
PCC: 79 chunks across 22 standards
AOP: 114 chunks across 31 standards
COP: 129 chunks across 29 standards
ASC: 50 chunks across 16 standards
QPS: 56 chunks across 12 standards
PCI: 106 chunks across 22 standards
GLD: 142 chunks across 34 standards
SQE: 128 chunks across 25 standards
MOI: 95 chunks across 17 standards
MPE: 23 chunks across 8 standards
HRP: 140 chunks across 10 standards
IPSG: 76 chunks across 14 standards
MMU: 85 chunks across 18 standards


## **3. Embedding model - local/open source**

In [45]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded.")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2277.16it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded.


## **4. Builing vector DB for each chapter**

In [46]:
jci_dbs = {}

for chapter, docs in chapter_chunks.items():
    if docs:
        print(f"Building FAISS for {chapter}...")
        jci_dbs[chapter] = FAISS.from_documents(docs, embeddings)

print("Built DBs for:", list(jci_dbs.keys()))

Building FAISS for UNKNOWN...
Building FAISS for FMS...
Building FAISS for APR...
Building FAISS for ACC...
Building FAISS for PCC...
Building FAISS for AOP...
Building FAISS for COP...
Building FAISS for ASC...
Building FAISS for QPS...
Building FAISS for PCI...
Building FAISS for GLD...
Building FAISS for SQE...
Building FAISS for MOI...
Building FAISS for MPE...
Building FAISS for HRP...
Building FAISS for IPSG...
Building FAISS for MMU...
Built DBs for: ['UNKNOWN', 'FMS', 'APR', 'ACC', 'PCC', 'AOP', 'COP', 'ASC', 'QPS', 'PCI', 'GLD', 'SQE', 'MOI', 'MPE', 'HRP', 'IPSG', 'MMU']


In [47]:
#Save indices per chapter
import os

BASE_SAVE_DIR = "faiss_jci_chapters"
os.makedirs(BASE_SAVE_DIR, exist_ok=True)

for chapter, db in jci_dbs.items():
    save_path = os.path.join(BASE_SAVE_DIR, chapter)
    db.save_local(save_path)

print("All chapter FAISS indices saved.")

All chapter FAISS indices saved.


In [48]:
#testing retrieval
test_query = "patient transfer communication"

results = jci_dbs["ACC"].similarity_search(test_query, k=3)

for i, r in enumerate(results):
    print(f"\nResult {i+1}")
    print(r.page_content[:400])
    print("-----")


Result 1
c) surgical and nonsurgical treatment services;
d) outpatient care programs; and
e) other organizations and other care settings.
The patient’s medical record(s) is a primary source of information on the care process and the patient’s progress 
and thus is an essential communication tool. For this information to be useful and to support the continuity 
of the patient’s care, it needs to be availabl
-----

Result 2
qualified staff and the correct medical equipment for the patient’s condition.
Measurable Elements of ACC.5
q 1. The hospital develops a transfer process that is based on criteria to address patients’ needs for con-
tinuing care.
q 2. The transfer process addresses how and when responsibility for continuing care is moved to another 
practitioner or setting and determines that the receiving organiz
-----

Result 3
information related to the patient be transferred with him or her so that medications and other treatments 
can continue uninterrupted, and the patient’s st

# 7 Gold Standards

In [49]:
from collections import Counter

# gold_standard = [
#     # ACC.1 — Screening for admission
#     {"query": "How should hospitals evaluate whether a patient's needs match available services?", "expected_standard": "ACC.1", "expected_chapter": "ACC"},
#     {"query": "What are the requirements for triaging patients with urgent or emergency needs?", "expected_standard": "ACC.1", "expected_chapter": "ACC"},
#     # ACC.1.1 — Delays in service
#     {"query": "What must hospitals do when there are delays in providing care to patients?", "expected_standard": "ACC.1.1", "expected_chapter": "ACC"},
#     # ACC.2 — Patient flow
#     {"query": "How should hospitals manage patient flow and bed allocation throughout the facility?", "expected_standard": "ACC.2", "expected_chapter": "ACC"},
#     # ACC.2.3 — Admission and discharge criteria
#     {"query": "What criteria should be used to admit or discharge patients from specialised units?", "expected_standard": "ACC.2.3", "expected_chapter": "ACC"},
#     # ACC.3 — Continuity of care
#     {"query": "How should information be shared between healthcare practitioners to ensure continuity?", "expected_standard": "ACC.3", "expected_chapter": "ACC"},
#     {"query": "What processes ensure care coordination when a patient moves between departments?", "expected_standard": "ACC.3", "expected_chapter": "ACC"},
#     # ACC.4 — Discharge planning
#     {"query": "When should discharge planning begin and what should it include?", "expected_standard": "ACC.4", "expected_chapter": "ACC"},
#     # ACC.4.1 — Patient education on discharge
#     {"query": "What instructions and information must be given to patients when they leave the hospital?", "expected_standard": "ACC.4.1", "expected_chapter": "ACC"},
#     # ACC.5 — Transfer
#     {"query": "What are the requirements for safely transferring a patient to another healthcare facility?", "expected_standard": "ACC.5", "expected_chapter": "ACC"},
#     {"query": "Who is responsible for monitoring a patient during transfer and what qualifications are needed?", "expected_standard": "ACC.5", "expected_chapter": "ACC"},
#     # ACC.5.1 — Transfer documentation
#     {"query": "What clinical information must be documented and handed over when transferring a patient?", "expected_standard": "ACC.5.1", "expected_chapter": "ACC"},
#     # ACC.6 — Transportation
#     {"query": "What safety and regulatory standards must hospital transportation services meet?", "expected_standard": "ACC.6", "expected_chapter": "ACC"},
#     {"query": "How should hospitals assess and arrange safe transport for patients being discharged?", "expected_standard": "ACC.6", "expected_chapter": "ACC"},
#     {"query": "What equipment and medications must be available in patient transport vehicles?", "expected_standard": "ACC.6", "expected_chapter": "ACC"},
# ]

gold_standard = [

    # ACC.1 — Screening and Admission
    {"query": "What screening criteria should be used when a patient arrives at the emergency department?", "expected_standard": "ACC.1", "expected_chapter": "ACC"},
    {"query": "How should hospitals handle patients with communicable diseases during screening?", "expected_standard": "ACC.1", "expected_chapter": "ACC"},

    # ACC.2.2 — Admission information
    {"query": "What information must be provided to patients when they are admitted to the hospital?", "expected_standard": "ACC.2.2", "expected_chapter": "ACC"},

    # ACC.2 — Patient Flow and Bed Management
    {"query": "How should hospitals manage patient flow to prevent emergency department overcrowding?", "expected_standard": "ACC.2", "expected_chapter": "ACC"},
    {"query": "What are the requirements for managing patients waiting for inpatient beds in the ED?", "expected_standard": "ACC.2", "expected_chapter": "ACC"},

    # ACC.2.3 — Intensive Care and Specialized Services
    {"query": "What criteria should be established for admission to intensive care units?", "expected_standard": "ACC.2.3", "expected_chapter": "ACC"},
    {"query": "How should hospitals determine when patients no longer need intensive care?", "expected_standard": "ACC.2.3", "expected_chapter": "ACC"},

    # ACC.3.1 — Continuity of Care
    {"query": "Who should be responsible for coordinating a patient's care throughout their hospital stay?", "expected_standard": "ACC.3.1", "expected_chapter": "ACC"},
    {"query": "What processes should be in place to ensure continuity when care responsibility is transferred between providers?", "expected_standard": "ACC.3.1", "expected_chapter": "ACC"},

    # ACC.4 — Discharge Planning
    {"query": "When should discharge planning begin for hospitalized patients?", "expected_standard": "ACC.4", "expected_chapter": "ACC"},

    # ACC.4.1 — Discharge Education
    {"query": "What education must be provided to patients and families before discharge?", "expected_standard": "ACC.4.1", "expected_chapter": "ACC"},

    # ACC.4.2 — Discharge Summary
    {"query": "What information must be included in a patient's discharge summary?", "expected_standard": "ACC.4.2", "expected_chapter": "ACC"},

    # ACC.4.2.1 — Emergency Care Documentation
    {"query": "What specific information must be documented for emergency department patients?", "expected_standard": "ACC.4.2.1", "expected_chapter": "ACC"},

    # ACC.4.3 — Complex Care Patients
    {"query": "How should hospitals manage information for patients with multiple complex diagnoses?", "expected_standard": "ACC.4.3", "expected_chapter": "ACC"},

    # ACC.4.4 — AMA (Against Medical Advice)
    {"query": "What procedures should be followed when a patient wants to leave against medical advice?", "expected_standard": "ACC.4.4", "expected_chapter": "ACC"},
    {"query": "What documentation is required when a patient leaves without notifying staff?", "expected_standard": "ACC.4.4.1", "expected_chapter": "ACC"},

    # ACC.5 — Patient Transfer
    {"query": "What criteria should be used to determine if a patient needs to be transferred to another facility?", "expected_standard": "ACC.5", "expected_chapter": "ACC"},

    # ACC.5.1 — Transfer Documentation
    {"query": "What information must be provided to the receiving hospital when transferring a patient?", "expected_standard": "ACC.5.1", "expected_chapter": "ACC"},

    # ACC.6 — Transportation Services
    {"query": "What are the hospital's responsibilities for patient transportation services?", "expected_standard": "ACC.6", "expected_chapter": "ACC"},

    # ACC.1.1 — Delays in Care
    {"query": "How should hospitals inform patients about delays in diagnostic or treatment services?", "expected_standard": "ACC.1.1", "expected_chapter": "ACC"},

    # Cross-standard queries (multi-label but mapped to primary)
    {"query": "What are all the documentation requirements when a patient is transferred from the ICU to another hospital?", "expected_standard": "ACC.5.1", "expected_chapter": "ACC"},
    {"query": "What education and follow-up is required for a complex care patient being discharged home?", "expected_standard": "ACC.4.1", "expected_chapter": "ACC"},

    # Edge cases
    {"query": "What are the requirements for psychiatric patient admission criteria?", "expected_standard": "ACC.2.3", "expected_chapter": "ACC"},
    {"query": "How should hospitals handle patients requiring organ transplant services?", "expected_standard": "ACC.2.3", "expected_chapter": "ACC"},
    {"query": "What infection control measures should be considered during patient transportation?", "expected_standard": "ACC.6", "expected_chapter": "ACC"},

]
print(f"Gold standard: {len(gold_standard)} queries")
standard_counts = Counter(item["expected_standard"] for item in gold_standard)
print("\nQueries per standard:")
for standard, count in sorted(standard_counts.items()):
    print(f"  {standard}: {count} queries")

Gold standard: 25 queries

Queries per standard:
  ACC.1: 2 queries
  ACC.1.1: 1 queries
  ACC.2: 2 queries
  ACC.2.2: 1 queries
  ACC.2.3: 4 queries
  ACC.3.1: 2 queries
  ACC.4: 1 queries
  ACC.4.1: 2 queries
  ACC.4.2: 1 queries
  ACC.4.2.1: 1 queries
  ACC.4.3: 1 queries
  ACC.4.4: 1 queries
  ACC.4.4.1: 1 queries
  ACC.5: 1 queries
  ACC.5.1: 2 queries
  ACC.6: 2 queries


# 8 Model definitions

In [67]:
# Models to compare
# We load one at a time during evaluation to avoid memory issues
MODELS_TO_EVALUATE = {
    "MiniLM (baseline)" : "sentence-transformers/all-MiniLM-L6-v2",
    "BGE-small"         : "BAAI/bge-small-en-v1.5",
    "BGE-large"         : "BAAI/bge-large-en-v1.5",
}

# k values to test
# hit@3 = did correct standard appear in top 3 results?
K_VALUES = [1, 3, 5]

print("Models to evaluate:")
for name, model_str in MODELS_TO_EVALUATE.items():
    print(f"  {name}: {model_str}")

Models to evaluate:
  MiniLM (baseline): sentence-transformers/all-MiniLM-L6-v2
  BGE-small: BAAI/bge-small-en-v1.5
  BGE-large: BAAI/bge-large-en-v1.5


# 9 Evaluation functions

In [68]:
import time

def run_evaluation(chapter_chunks, gold_standard, model_name, model_string, k_values):
    print(f"\n{'='*50}")
    print(f"Evaluating: {model_name}")
    print(f"{'='*50}")

    # Load embedding model
    print("Loading model...")
    embeddings = HuggingFaceEmbeddings(
        model_name=model_string,
        encode_kwargs={"normalize_embeddings": True}
    )

    # Build FAISS DBs only for chapters we need
    needed_chapters = set(item["expected_chapter"] for item in gold_standard)
    print(f"Building DBs for: {needed_chapters}")
    dbs = {}
    for chapter in needed_chapters:
        if chapter in chapter_chunks and chapter_chunks[chapter]:
            dbs[chapter] = FAISS.from_documents(chapter_chunks[chapter], embeddings)
            print(f"  Built {chapter} DB")

    # Run every query
    results = []
    max_k = max(k_values)

    for item in gold_standard:
        query             = item["query"]
        expected_standard = item["expected_standard"]
        chapter           = item["expected_chapter"]

        db = dbs.get(chapter)
        if db is None:
            continue

        retrieved       = db.similarity_search(query, k=max_k)
        retrieved_codes = [r.metadata["standard_code"] for r in retrieved]

        # Slice per k
        topk_results = {}
        for k in k_values:
            topk_results[f"top{k}"] = retrieved_codes[:k]

        row = {
            "model":             model_name,
            "query":             query,
            "expected_standard": expected_standard,
            "top1_codes": topk_results["top1"],
            "top3_codes": topk_results["top3"],
            "top5_codes": topk_results["top5"],
            }
        # Check if expected standard appears in top k results
        for k in k_values:
            #row[f"hit@{k}"] = expected_standard in retrieved_codes[:k]
            row[f"hit@{k}"] = expected_standard in topk_results[f"top{k}"]

        results.append(row)

    # Print summary
    print(f"\nResults for {model_name}:")
    for k in k_values:
        hits     = sum(1 for r in results if r[f"hit@{k}"])
        hit_rate = hits / len(results) * 100
        print(f"  Hit Rate @{k}: {hit_rate:.0f}% ({hits}/{len(results)})")

    return results

print("Evaluation function defined.")

Evaluation function defined.


# 10 Running evalutation

In [69]:
import pandas as pd

all_results = []

for model_name, model_string in MODELS_TO_EVALUATE.items():
    start   = time.time()
    results = run_evaluation(
        chapter_chunks=chapter_chunks,
        gold_standard=gold_standard,
        model_name=model_name,
        model_string=model_string,
        k_values=K_VALUES
    )
    duration = round(time.time() - start, 1)
    print(f"  Time taken: {duration}s")
    all_results.extend(results)

print("\n✓ All models evaluated.")


Evaluating: MiniLM (baseline)
Loading model...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2176.12it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Building DBs for: {'ACC'}
  Built ACC DB

Results for MiniLM (baseline):
  Hit Rate @1: 76% (19/25)
  Hit Rate @3: 88% (22/25)
  Hit Rate @5: 92% (23/25)
  Time taken: 6.3s

Evaluating: BGE-small
Loading model...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2023.80it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Building DBs for: {'ACC'}
  Built ACC DB

Results for BGE-small:
  Hit Rate @1: 68% (17/25)
  Hit Rate @3: 80% (20/25)
  Hit Rate @5: 88% (22/25)
  Time taken: 7.6s

Evaluating: BGE-large
Loading model...


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 1781.94it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Building DBs for: {'ACC'}
  Built ACC DB

Results for BGE-large:
  Hit Rate @1: 84% (21/25)
  Hit Rate @3: 88% (22/25)
  Hit Rate @5: 88% (22/25)
  Time taken: 19.3s

✓ All models evaluated.


# 11 Summary table

In [70]:
df_all = pd.DataFrame(all_results)

summary_rows = []
for model_name in MODELS_TO_EVALUATE.keys():
    model_df = df_all[df_all["model"] == model_name]
    row = {"Model": model_name}
    for k in K_VALUES:
        hits     = model_df[f"hit@{k}"].sum()
        total    = len(model_df)
        hit_rate = hits / total * 100
        row[f"Hit Rate @{k}"] = f"{hit_rate:.0f}% ({hits}/{total})"
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
print("=== SUMMARY: Hit Rate by Model ===")
print(summary_df.to_string(index=False))

=== SUMMARY: Hit Rate by Model ===
            Model Hit Rate @1 Hit Rate @3 Hit Rate @5
MiniLM (baseline) 76% (19/25) 88% (22/25) 92% (23/25)
        BGE-small 68% (17/25) 80% (20/25) 88% (22/25)
        BGE-large 84% (21/25) 88% (22/25) 88% (22/25)


# 12 Where each model failed

In [72]:
print("=== MISSES AT @3 ===")

misses = df_all[df_all["hit@3"] == False][
    ["model", "query", "expected_standard", "top1_codes", "top3_codes", "top5_codes"]
]

if len(misses) == 0:
    print("All queries hit at @3 for all models!")
else:
    for _, row in misses.iterrows():
        print(f"\nModel:    {row['model']}")
        print(f"Query:    {row['query']}")
        print(f"Expected: {row['expected_standard']}")
        print(f"Top1:     {row['top1_codes']}")
        print(f"Top3:     {row['top3_codes']}")
        print(f"Top5:     {row['top5_codes']}")

=== MISSES AT @3 ===

Model:    MiniLM (baseline)
Query:    What processes should be in place to ensure continuity when care responsibility is transferred between providers?
Expected: ACC.3.1
Top1:     ['ACC.3']
Top3:     ['ACC.3', 'ACC.3', 'ACC.3']
Top5:     ['ACC.3', 'ACC.3', 'ACC.3', 'ACC.3', 'ACC.3.1']

Model:    MiniLM (baseline)
Query:    What specific information must be documented for emergency department patients?
Expected: ACC.4.2.1
Top1:     ['ACC.4.2']
Top3:     ['ACC.4.2', 'ACC.4.3', 'ACC.4.2']
Top5:     ['ACC.4.2', 'ACC.4.3', 'ACC.4.2', 'ACC.2.3', 'ACC.2.2']

Model:    MiniLM (baseline)
Query:    What documentation is required when a patient leaves without notifying staff?
Expected: ACC.4.4.1
Top1:     ['ACC.4.4']
Top3:     ['ACC.4.4', 'ACC.4.4', 'ACC.3']
Top5:     ['ACC.4.4', 'ACC.4.4', 'ACC.3', 'ACC.4.4', 'ACC.4.4']

Model:    BGE-small
Query:    How should hospitals determine when patients no longer need intensive care?
Expected: ACC.2.3
Top1:     ['ACC.4.4']
Top3:    

# 13 Save results

In [73]:
# Save summary
summary_df.to_csv("evaluation_summary.csv", index=False)
print("Saved: evaluation_summary.csv")

# Save detailed results
df_save = df_all.copy()
#df_save["retrieved_codes"] = df_save["retrieved_codes"].apply(lambda x: ", ".join(x))
for col in ["top1_codes", "top3_codes", "top5_codes"]:
    df_save[col] = df_save[col].apply(lambda x: ", ".join(x))
    
df_save.to_csv("evaluation_detailed.csv", index=False)
print("Saved: evaluation_detailed.csv")


Saved: evaluation_summary.csv
Saved: evaluation_detailed.csv


# Diagnostic checks

In [56]:
# Check what heading word each problem chapter uses
for chapter in ["APR", "AOP", "COP", "HRP"]:
    full_text = "\n".join([p.page_content for p in chapter_pages[chapter]])
    
    # Find lines containing the chapter code followed by a number
    matches = re.findall(rf'.{{0,30}}{chapter}\.\d+.{{0,30}}', full_text)
    
    print(f"\n=== {chapter} — sample matches ===")
    for m in matches[:10]:  # show first 10
        print(f"  {repr(m)}")


=== APR — sample matches ===
  '• APR.8, Rationale: Added an example '
  'cess to authorized JCI staff (APR.4), will lead to the loss or de'
  'APR.1  The hospital meets all requi'
  'APR.2  The hospital provides JCI wi'
  'APR.3  The hospital reports within '
  'APR.4  The hospital permits on-site'
  'APR.5  The hospital allows JCI to r'
  'APR.7 The hospital selects and uses'
  'APR.8 The hospital accurately repre'
  'APR.9 Any individual hospital staff'

=== AOP — sample matches ===
  'Intent of AOP.1'
  'Measurable Elements of AOP.1'
  'Intent of AOP.1.1'
  'Measurable Elements of AOP.1.1'
  'Intent of AOP.1.2 and AOP.1.2.1'
  'Measurable Elements of AOP.1.2'
  'Measurable Elements of AOP.1.2.1'
  'Intent of AOP.1.3'
  'Measurable Elements of AOP.1.3'
  'Intent of AOP.1.3.1'

=== COP — sample matches ===
  'Intent of COP.1'
  'Measurable Elements of COP.1'
  'Intent of COP.2'
  'Measurable Elements of COP.2'
  'Intent of COP.2.1'
  'Measurable Elements of COP.2.1'
  'Intent of COP

In [57]:
# Look for the actual heading line in AOP
for chapter in ["AOP", "COP", "HRP"]:
    full_text = "\n".join([p.page_content for p in chapter_pages[chapter]])
    
    # Search specifically for "Standard CHAPTER.X" pattern
    matches = re.findall(
        rf'(?:Standard|Requirement)\s+{chapter}\.\d+(?:\.\d+)?', 
        full_text
    )
    print(f"\n{chapter} — headings found: {len(matches)}")
    for m in matches[:5]:
        print(f"  {repr(m)}")


AOP — headings found: 0

COP — headings found: 0

HRP — headings found: 0


In [58]:
for chapter in ["APR", "AOP", "COP", "HRP"]:
    full_text = "\n".join([p.page_content for p in chapter_pages[chapter]])
    
    # Find the FIRST occurrence of the chapter code with a number
    # and show 300 characters before and after it
    match = re.search(rf'{chapter}\.\d+', full_text)
    if match:
        start = max(0, match.start() - 300)
        end   = min(len(full_text), match.end() + 300)
        print(f"\n{'='*50}")
        print(f"{chapter} — context around first code:")
        print(repr(full_text[start:end]))


APR — context around first code:
'JCI ACCredItAtIon StAndArdS for HoSpItAlS, 7tH edItIon. SpeCIAl edItIon for SIngApore MoH\nx\nAccreditation Participation Requirements (APR)\n• APR.8, Rationale: Added an example of inappropriate representation of accreditation status\n• Made minor editorial revisions throughout the chapter\nInternational Patient Safety Goals (IPSG)\n• IPSG.1\no Intent: Revised to address the labeling of elements associated with patient care (for example, \ndietary t'

AOP — context around first code:
'Ion StAndArdS for HoSpItAlS, 7tH edItIon. SpeCIAl edItIon for SIngApore MoH\n82\nStandards, Intents, and Measurable Elements\nStandard AOP .1\nAll patients cared for by the hospital have their health care needs identified through an assessment process that \nhas been defined by the hospital. 𝖯\nIntent of AOP.1\nAn effective patient-assessment process results in decisions about the patient’s immediate and continuing \ntreatment needs for emergency, elective, or planned c

In [59]:
# Find what APR standard headings actually look like
apr_text = "\n".join([p.page_content for p in chapter_pages["APR"]])

# Search for lines that start with APR. followed by a number
for match in re.finditer(r'(?:\n|^)(APR\.\d+(?:\.\d+)?)\s+\w', apr_text):
    start = max(0, match.start() - 100)
    end   = min(len(apr_text), match.end() + 200)
    print(f"\n--- Found: {match.group(1)} ---")
    print(repr(apr_text[start:end]))
    print()


--- Found: APR.1 ---
'gnizes the JCI Direct Connect website as the \nofficial location for the posting of all current APRs.\nAPR.1  The hospital meets all requirements for timely submissions of data and information to Joint \nCommission International (JCI).\nAPR.2  The hospital provides JCI with accurate and complete information thro'


--- Found: APR.2 ---
'equirements for timely submissions of data and information to Joint \nCommission International (JCI).\nAPR.2  The hospital provides JCI with accurate and complete information throughout all phases of the \naccreditation process.\nAPR.3  The hospital reports within 30 days of the effective date of any change(s) i'


--- Found: APR.3 ---
'ides JCI with accurate and complete information throughout all phases of the \naccreditation process.\nAPR.3  The hospital reports within 30 days of the effective date of any change(s) in the hospital’s profile \n(electronic database) or information provided to JCI via the E-App before and between 

In [60]:
# Investigate HRP
hrp_docs = chapter_chunks["HRP"]

print(f"HRP total chunks: {len(hrp_docs)}")
print(f"\nChunks per standard:")
unique_codes = sorted(set(doc.metadata["standard_code"] for doc in hrp_docs))
for code in unique_codes:
    count = sum(1 for doc in hrp_docs if doc.metadata["standard_code"] == code)
    print(f"  {code}: {count} chunks")

# Check the biggest standard
biggest = max(unique_codes, key=lambda c: sum(1 for d in hrp_docs if d.metadata["standard_code"] == c))
print(f"\nBiggest standard: {biggest}")
biggest_chunks = [d for d in hrp_docs if d.metadata["standard_code"] == biggest]
print(f"First chunk preview:")
print(biggest_chunks[0].page_content[:300])
print(f"\nLast chunk preview:")
print(biggest_chunks[-1].page_content[:300])

HRP total chunks: 140

Chunks per standard:
  HRP.1.1: 4 chunks
  HRP.2: 2 chunks
  HRP.3: 3 chunks
  HRP.3.1: 2 chunks
  HRP.4: 3 chunks
  HRP.5: 2 chunks
  HRP.6: 4 chunks
  HRP.7: 1 chunks
  HRP.7.1: 118 chunks
  INTRO: 1 chunks

Biggest standard: HRP.7.1
First chunk preview:
The hospital informs patients and families about how to gain access to clinical research, clinical investigations, 
or clinical trials and includes protections for vulnerable populations to minimize potential coercion or undue 
influence. 𝖯
Intent of HRP.7 and HRP.7.1
A hospital that conducts clinic

Last chunk preview:
two drugs, usually unnecessarily, from the same 
therapeutic category at the same time.
time-out A pause, just prior to performing a 
surgical or other procedure, during which the entire 
surgical or procedural team verifies the correct 
patient, procedure, or site. Even when there is only 
one pers


In [61]:
# Check what's at the end of HRP chapter pages
hrp_pages = chapter_pages["HRP"]
print(f"HRP has {len(hrp_pages)} pages")
print("\n--- Last page of HRP ---")
print(hrp_pages[-1].page_content[:500])
print("\n--- Second to last page ---")
print(hrp_pages[-2].page_content[:500])

HRP has 32 pages

--- Last page of HRP ---
JCI ACCredItAtIon StAndArdS for HoSpItAlS, 7tH edItIon. SpeCIAl edItIon for SIngApore MoH
372

--- Second to last page ---
JCI ACCredItAtIon StAndArdS for HoSpItAlS, 7tH edItIon. SpeCIAl edItIon for SIngApore MoH
370
members; leadership and governance; vendors and 
contracted employees; members of the community; 
and others.
standard A statement that defines the 
performance expectations, structures, or processes 
that must be in place for an organization to provide 
safe and high-quality patient care, treatment, and 
services.
standardized measure A performance measure 
that has precisely defined specifications, ha


In [62]:
# Demonstrate the old cross-contamination bug
# Using the ORIGINAL broken pattern that matched any chapter code anywhere

print("=== DEMONSTRATING OLD BUG — CROSS-CONTAMINATION ===")
print("Using old pattern: r'([A-Z]{3,4}\\.\\d+)' — matches ANY code anywhere\n")

# Simulate old chunking on just ACC chapter
acc_text = "\n".join([p.page_content for p in chapter_pages["ACC"]])

# Old pattern — no chapter restriction
old_pattern = r'(?:\n|^)([A-Z]{3,4}\.\d+(?:\.\d+)?)\b'
old_splits = re.split(old_pattern, acc_text)

old_codes = []
for i in range(1, len(old_splits), 2):
    code = old_splits[i].strip()
    old_codes.append(code)

# Count foreign codes
from collections import Counter
code_counts = Counter(old_codes)
foreign = {code: count for code, count in code_counts.items() 
           if not code.startswith("ACC")}
acc_codes = {code: count for code, count in code_counts.items() 
             if code.startswith("ACC")}


for code, count in sorted(acc_codes.items()):
    print(f"  {code}: {count} chunks")

for code, count in sorted(foreign.items()):
    print(f"  {code}: {count} chunks")

print(f"\nWith NEW pattern (Standard ACC.X): 0 foreign codes")

=== DEMONSTRATING OLD BUG — CROSS-CONTAMINATION ===
Using old pattern: r'([A-Z]{3,4}\.\d+)' — matches ANY code anywhere

  ACC.1: 1 chunks
  ACC.1.1: 1 chunks
  ACC.2: 1 chunks
  ACC.2.1: 1 chunks
  ACC.2.2: 1 chunks
  ACC.2.3: 1 chunks
  ACC.3: 1 chunks
  ACC.3.1: 1 chunks
  ACC.4: 1 chunks
  ACC.4.1: 1 chunks
  ACC.4.2: 2 chunks
  ACC.4.3: 1 chunks
  ACC.4.4: 2 chunks
  ACC.5: 1 chunks
  ACC.5.1: 1 chunks
  ACC.6: 1 chunks
  GLD.3.1: 1 chunks
  MOI.4: 2 chunks
  MOI.8: 1 chunks

With NEW pattern (Standard ACC.X): 0 foreign codes


In [63]:
# Demonstrate old cross-contamination bug
acc_text = "\n".join([p.page_content for p in chapter_pages["ACC"]])

old_pattern = r'(?:\n|^)([A-Z]{3,4}\.\d+(?:\.\d+)?)\b'
old_splits = re.split(old_pattern, acc_text)

old_codes = []
for i in range(1, len(old_splits), 2):
    old_codes.append(old_splits[i].strip())

from collections import Counter
code_counts = Counter(old_codes)

print("FOREIGN codes in ACC with old pattern:")
for code, count in sorted(code_counts.items()):
    if not code.startswith("ACC"):
        print(f"  {code}: {count} chunks")

print("\nACC codes with old pattern:")
for code, count in sorted(code_counts.items()):
    if code.startswith("ACC"):
        print(f"  {code}: {count} chunks")

print("\nWith new pattern — foreign codes: 0")

FOREIGN codes in ACC with old pattern:
  GLD.3.1: 1 chunks
  MOI.4: 2 chunks
  MOI.8: 1 chunks

ACC codes with old pattern:
  ACC.1: 1 chunks
  ACC.1.1: 1 chunks
  ACC.2: 1 chunks
  ACC.2.1: 1 chunks
  ACC.2.2: 1 chunks
  ACC.2.3: 1 chunks
  ACC.3: 1 chunks
  ACC.3.1: 1 chunks
  ACC.4: 1 chunks
  ACC.4.1: 1 chunks
  ACC.4.2: 2 chunks
  ACC.4.3: 1 chunks
  ACC.4.4: 2 chunks
  ACC.5: 1 chunks
  ACC.5.1: 1 chunks
  ACC.6: 1 chunks

With new pattern — foreign codes: 0


In [64]:
# Show that HRP's last pages were glossary content before the fix
hrp_pages = chapter_pages["HRP"]

print(f"HRP total pages after fix: {len(hrp_pages)}")
print(f"\n--- Last page of HRP ---")
print(hrp_pages[-1].page_content[:400])

print(f"\n--- Second to last page ---")
print(hrp_pages[-2].page_content[:400])

# Now manually check what the glossary pages looked like
# by loading all pages and finding the ones we stripped
loader_check = PyPDFLoader(JCI_PDF)
all_pages = loader_check.load()

# Rebuild chapter assignment WITHOUT the glossary fix to show the bug
chapter_pages_old = defaultdict(list)
current = "UNKNOWN"
for page in all_pages:
    detected = detect_chapter_from_text(page.page_content)
    if detected:
        current = detected
    chapter_pages_old[current].append(page)

old_hrp = chapter_pages_old["HRP"]
print(f"\n=== BEFORE FIX: HRP had {len(old_hrp)} pages ===")
print(f"=== AFTER FIX:  HRP has {len(hrp_pages)} pages ===")
print(f"=== Pages removed: {len(old_hrp) - len(hrp_pages)} ===")

print(f"\n--- These are the pages that were incorrectly assigned to HRP ---")
for i, page in enumerate(old_hrp):
    if is_end_of_book_page(page.page_content):
        print(f"\nPage {i+1} of old HRP (glossary/appendix content):")
        print(page.page_content[:300])
        print("-----")

HRP total pages after fix: 32

--- Last page of HRP ---
JCI ACCredItAtIon StAndArdS for HoSpItAlS, 7tH edItIon. SpeCIAl edItIon for SIngApore MoH
372

--- Second to last page ---
JCI ACCredItAtIon StAndArdS for HoSpItAlS, 7tH edItIon. SpeCIAl edItIon for SIngApore MoH
370
members; leadership and governance; vendors and 
contracted employees; members of the community; 
and others.
standard A statement that defines the 
performance expectations, structures, or processes 
that must be in place for an organization to provide 
safe and high-quality patient care, treatment, and 

=== BEFORE FIX: HRP had 45 pages ===
=== AFTER FIX:  HRP has 32 pages ===
=== Pages removed: 13 ===

--- These are the pages that were incorrectly assigned to HRP ---

Page 9 of old HRP (glossary/appendix content):
JCI ACCredItAtIon StAndArdS for HoSpItAlS, 7tH edItIon. SpeCIAl edItIon for SIngApore MoH
336
References
Council for International Organizations of Medical Sciences. 
International Ethical Guidelines for 